# 模型统一配置说明

## 配置一览

模型与密钥统一放在仓库根目录 `.env`，Notebook 用 `load_dotenv(find_dotenv())` 读取，代码里不写死地址与密钥。

| 能力 | 服务 | 环境变量 | 模型 |
| --- | --- | --- | --- |
| 大语言模型 | vLLM（OpenAI 兼容 `/v1`） | `vLLM_QWEN_BASE_URL` / `vLLM_QWEN_API_KEY` / `vLLM_QWEN_MODEL_ID` | `my_qwen3.8_flash_next` |
| Embedding | Xinference（`/v1/embeddings`） | `XINFERENCE_OPENAI_BASE_URL` / `XINFERENCE_API_KEY` / `XINFERENCE_EMBEDDING_MODEL_ID` | `my_qwen3_embedding_0.6b`（1024 维） |
| Reranker | Xinference（`/v1/rerank`） | `XINFERENCE_OPENAI_BASE_URL` / `XINFERENCE_API_KEY` / `XINFERENCE_RERANKER_MODEL_ID` | `my_qwen3_reranker_0.6b` |

Embedding 与 Reranker 由同一个 Xinference 实例提供：`XINFERENCE_BASE_URL` 是原生 REST 根地址（给 REST 客户端用），`XINFERENCE_OPENAI_BASE_URL` 是它的 `/v1` 兼容地址（给 OpenAI 风格客户端用），两者共用 `XINFERENCE_API_KEY`。

本 Notebook 用 `env_rag_uv` 内核（`~/env_rag_uv`）实跑通过，每段代码后面的 **预期输出** 就是实跑结果；大语言模型的回答是生成式的，字句会略有出入，意思一致即可。


代理清除，如果遇到代理冲突报错

In [1]:
import warnings
warnings.filterwarnings("ignore")


import os
for k in [
    "HTTP_PROXY",
    "HTTPS_PROXY",
    "ALL_PROXY",
    "http_proxy",
    "https_proxy",
    "all_proxy"
]:
    os.environ.pop(k, None)


## 大语言模型：vLLM（OpenAI 兼容）

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

_ = load_dotenv(find_dotenv())

client = OpenAI(
    base_url=os.getenv("vLLM_QWEN_BASE_URL"),
    api_key=os.getenv("vLLM_QWEN_API_KEY"),
)

response = client.chat.completions.create(
    model=os.getenv("vLLM_QWEN_MODEL_ID"),
    messages=[
        {"role": "system", "content": "你是一个 RAG 助教"},
        {"role": "user", "content": "用一句话说明什么是 RAG"}
    ],
    temperature=0.3,
    max_tokens=256,
    stream=False,
)

print(response.choices[0].message.content)




RAG（检索增强生成）是一种让大模型在回答问题前先从外部知识库检索相关资料，再基于这些资料生成答案的技术。


**预期输出：**

```text
RAG（检索增强生成）是一种让大模型在回答问题前先从外部知识库检索相关资料，再基于这些资料生成答案的技术。
```

回答由模型生成，字句会略有不同，意思一致即为通过。


In [3]:
import os
import httpx
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model

_ = load_dotenv(find_dotenv())

my_llm = init_chat_model(
    model=os.getenv("vLLM_QWEN_MODEL_ID"),
    model_provider="openai",
    base_url=os.getenv("vLLM_QWEN_BASE_URL"),
    api_key=os.getenv("vLLM_QWEN_API_KEY"),
    http_client=httpx.Client(trust_env=False),
    temperature=0.3,
    max_tokens=4096,
)

print(my_llm.invoke("用一句话说明什么是 RAG").content)




RAG（检索增强生成）是一种让大模型在回答问题前先从外部知识库检索相关资料，再基于这些资料生成更准确、可溯源答案的技术。


**预期输出：**

```text
RAG（检索增强生成）是一种让大模型在回答问题前先从外部知识库检索相关资料，再基于这些资料生成更准确、可溯源答案的技术。
```

同上，回答内容与上一段一致即可；`init_chat_model` 让后续课件可以只换模型不改代码。


## Embedding 模型：Xinference（OpenAI 兼容 `/v1/embeddings`）

In [4]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import OpenAIEmbeddings

_ = load_dotenv(find_dotenv())

my_emb = OpenAIEmbeddings(
    base_url=os.getenv("XINFERENCE_OPENAI_BASE_URL"),
    api_key=os.getenv("XINFERENCE_API_KEY"),
    model=os.getenv("XINFERENCE_EMBEDDING_MODEL_ID"),
)

vec = my_emb.embed_query("人工智能是什么")

print(len(vec))
print(vec[:5])


1024
[-0.048095703125, -0.032470703125, -0.0081787109375, -0.0810546875, -0.0556640625]


**预期输出：**

```text
1024
[-0.048095703125, -0.032470703125, -0.0081787109375, -0.0810546875, -0.0556640625]
```

维度必须与 Milvus collection 的向量维度一致（这里 1024），换 Embedding 模型要重建 collection。


## Reranker 模型：Xinference（OpenAI 兼容 `/v1/rerank`）

In [5]:
import os
import httpx
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

query = "什么是人工智能"
documents = [
    "人工智能是研究如何让机器模拟人类智能",
    "今天的天气很好",
]

# Xinference 暴露 OpenAI 兼容的 /v1 接口，rerank 走 /v1/rerank
resp = httpx.post(
    os.environ["XINFERENCE_OPENAI_BASE_URL"].rstrip("/") + "/rerank",
    headers={"Authorization": "Bearer " + os.environ["XINFERENCE_API_KEY"]},
    json={
        "model": os.environ["XINFERENCE_RERANKER_MODEL_ID"],
        "query": query,
        "documents": documents,
        "top_n": len(documents),
    },
    timeout=60,
)
resp.raise_for_status()

for item in resp.json()["results"]:
    print(item["index"], round(item["relevance_score"], 6), documents[item["index"]])


0 1.0 人工智能是研究如何让机器模拟人类智能
1 9e-05 今天的天气很好


**预期输出：**

```text
0 1.0 人工智能是研究如何让机器模拟人类智能
1 9e-05 今天的天气很好
```

相关文档的 `relevance_score` 明显高于无关文档（1.0 对 9e-05），说明 Reranker 生效。


## 备选：Xinference 原生 REST 客户端

In [6]:
# 备选写法：Xinference 原生 REST 客户端（依赖 xinference-client，本项目锁 1.13.0）
import os
from dotenv import load_dotenv, find_dotenv
from xinference_client import RESTfulClient

_ = load_dotenv(find_dotenv())

xf = RESTfulClient(
    os.environ["XINFERENCE_BASE_URL"],
    api_key=os.environ["XINFERENCE_API_KEY"],
)

for model_id, info in xf.list_models().items():
    print(model_id, info["model_type"], info.get("dimensions"))

reranker = xf.get_model(os.environ["XINFERENCE_RERANKER_MODEL_ID"])
result = reranker.rerank(documents=documents, query=query)
print([(item["index"], round(item["relevance_score"], 6)) for item in result["results"]])


my_qwen3_reranker_0.6b rerank None
my_qwen3_embedding_0.6b embedding 1024


[(0, 1.0), (1, 9e-05)]


**预期输出：**

```text
my_qwen3_reranker_0.6b rerank None
my_qwen3_embedding_0.6b embedding 1024
[(0, 1.0), (1, 9e-05)]
```

原生客户端与 `/v1` 兼容接口看到的是同一批模型；`xinference-client` 1.13 起模块名是 `xinference_client`，类名 `RESTfulClient`。
